# Electron g-2 Parameter Sweep — Automatic Tuning

**Just edit the ranges and number of trials in the first cell, then Run All.**
It will try many combinations and give you a clean sorted table of the best results.

In [ ]:
import cupy as cp
import numpy as np
from tqdm import tqdm
import time
import pandas as pd
import random

# ====================== SWEEP SETTINGS (edit these) ======================
N_paths_total = 500_000_000
BATCH_SIZE    = 50_000_000

# Number of random trials to run (50–200 is good balance of speed vs coverage)
N_TRIALS = 80

# Parameter ranges (feel free to widen/narrow)
walk_length_range   = [20, 24, 28, 32, 36]
phase_scale_range   = [20, 25, 30, 35, 40, 45, 50]
fbs_power_range     = [4.0, 5.0, 6.0, 7.0, 8.0]

C = 9.6e-7
alpha = 1 / 137.035999084

print(f"Starting sweep: {N_TRIALS} random trials")
print(f"Walk lengths: {walk_length_range}")
print(f"Phase scales: {phase_scale_range}")
print(f"FBS powers: {fbs_power_range}")

In [ ]:
# ====================== FIXED SETUP (run once) ======================
phi = (1 + np.sqrt(5)) / 2
inv_phi = 1 / phi

def generate_600cell_vertices():
    coords = []
    for signs in np.array(np.meshgrid(*[[-0.5,0.5]]*4)).T.reshape(-1,4):
        coords.append(signs)
    for i in range(4):
        for s in [-1,1]:
            v = np.zeros(4); v[i] = s; coords.append(v)
    base = [0.5*phi, 0.5, 0.5*inv_phi, 0]
    for perm in [[0,1,2,3],[0,1,3,2],[0,2,1,3],[0,2,3,1],[0,3,1,2],[0,3,2,1]]:
        for signs in [[1,1,1],[1,1,-1],[1,-1,1],[-1,1,1]]:
            v = np.zeros(4)
            v[perm[0]] = signs[0] * base[0]
            v[perm[1]] = signs[1] * base[1]
            v[perm[2]] = signs[2] * base[2]
            coords.append(v)
    verts = np.unique(np.round(coords, decimals=10), axis=0)
    verts = verts / np.linalg.norm(verts, axis=1)[:, None]
    return cp.asarray(verts, dtype=cp.float32)

vertices = generate_600cell_vertices()
N_VERT = vertices.shape[0]

dists = cp.linalg.norm(vertices[:, None] - vertices[None, :], axis=-1)
edge_threshold = cp.sort(dists.flatten())[720*2 + 1]
adj_mask = (dists < edge_threshold + 1e-6) & (dists > 1e-6)
neighbors = [cp.argwhere(adj_mask[i])[:,0] for i in range(N_VERT)]

n_batches = (N_paths_total + BATCH_SIZE - 1) // BATCH_SIZE

In [ ]:
# ====================== SWEEP LOOP ======================
results = []
start_time = time.time()

for trial in tqdm(range(N_TRIALS), desc="Sweeping parameters"):
    # Randomly pick parameters for this trial
    walk_len   = random.choice(walk_length_range)
    phase_sc   = random.choice(phase_scale_range)
    fbs_pwr    = random.choice(fbs_power_range)

    raw_amplitudes = cp.zeros(3, dtype=cp.complex64)

    for batch_idx in range(n_batches):
        batch_size = min(BATCH_SIZE, N_paths_total - batch_idx * BATCH_SIZE)
        current = cp.random.randint(0, N_VERT, batch_size)
        phase = cp.zeros(batch_size, dtype=cp.complex64)
        fbs_grade = cp.ones(batch_size, dtype=cp.float32)

        for step in range(walk_len):
            neigh_choices = cp.array([cp.random.choice(n, size=1)[0] for n in neighbors])[current]
            edge_len = cp.linalg.norm(vertices[current] - vertices[neigh_choices], axis=1)
            phase += 1j * phase_sc * edge_len * fbs_grade
            fbs_grade *= 1.0 / (edge_len ** fbs_pwr + 0.1)
            current = neigh_choices

        amp = cp.exp(phase) * fbs_grade
        e_amp = amp * cp.exp(1j * 0.0)
        h_amp = amp * cp.exp(1j * np.pi / 2)
        q_amp = amp * cp.exp(1j * np.pi)

        raw_amplitudes[0] += cp.sum(e_amp)
        raw_amplitudes[1] += cp.sum(h_amp)
        raw_amplitudes[2] += cp.sum(q_amp)

    # Compute metrics
    interference = raw_amplitudes / N_paths_total
    abs_interf = cp.abs(interference)
    total_mag = cp.sum(abs_interf)
    mean_amp = float(cp.mean(cp.abs(amp)))

    S = alpha / (2 * np.pi) * mean_amp
    mixing_sum = float(abs_interf[2] / total_mag) + 0.7 * float(abs_interf[1] / total_mag)
    delta_mu_e = C * mixing_sum * S

    results.append({
        'trial': trial,
        'walk_length': walk_len,
        'phase_scale': phase_sc,
        'fbs_power': fbs_pwr,
        'mean_amp': mean_amp,
        'S': S,
        'delta_mu_e': delta_mu_e,
        'raw_sum_mag': float(cp.abs(raw_amplitudes).sum())
    })

# ====================== RESULTS ======================
df = pd.DataFrame(results)
df = df.sort_values('delta_mu_e')

print("\n=== BEST CONFIGURATIONS (lowest δμ_e) ===")
print(df.head(15).to_string(index=False))

print(f"\nTotal sweep time: {time.time() - start_time:.1f} seconds")
df.to_csv("sweep_results.csv", index=False)
print("Results saved to sweep_results.csv")